# TIỀN XỬ LÝ DỮ LIỆU NGUỒN CHO KHO DỮ LIỆU BÓNG ĐÁ
**Đề tài:** Phân tích hiệu suất thi đấu cầu thủ và hiệu quả vận hành CLB Bóng đá Châu Âu
**Quy trình 6 bước tiền xử lý dữ liệu**

In [ ]:
# 1. Import các thư viện cần thiết
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
print('Import các thư viện thành công.')

In [ ]:
# 2. Đọc dữ liệu từ các file CSV gốc
DATA_DIR = 'data' if os.path.exists('data') else os.path.join('football', 'data')

df_competitions = pd.read_csv(os.path.join(DATA_DIR, 'competitions.csv'), low_memory=False)
df_clubs        = pd.read_csv(os.path.join(DATA_DIR, 'clubs.csv'),        low_memory=False)
df_players      = pd.read_csv(os.path.join(DATA_DIR, 'players.csv'),      low_memory=False)
df_games        = pd.read_csv(os.path.join(DATA_DIR, 'games.csv'),        low_memory=False)
df_appearances  = pd.read_csv(os.path.join(DATA_DIR, 'appearances.csv'),  low_memory=False)

print('Kích thước dữ liệu gốc ban đầu:')
print(f'  - competitions: {df_competitions.shape[0]:,} dòng, {df_competitions.shape[1]} cột')
print(f'  - clubs       : {df_clubs.shape[0]:,} dòng, {df_clubs.shape[1]} cột')
print(f'  - players     : {df_players.shape[0]:,} dòng, {df_players.shape[1]} cột')
print(f'  - games       : {df_games.shape[0]:,} dòng, {df_games.shape[1]} cột')
print(f'  - appearances : {df_appearances.shape[0]:,} dòng, {df_appearances.shape[1]} cột')

In [ ]:
# 3. Khai báo tham số phạm vi (Scope)
EXPANDED_COMPETITIONS = [
    'GB1', 'ES1', 'IT1', 'L1', 'FR1',
    'PO1', 'NL1', 'TR1', 'BE1', 'SC1',
    'CL', 'EL', 'UCOL'
]
EXPANDED_SEASONS = [2022, 2023]

print(f'Phạm vi giải đấu ({len(EXPANDED_COMPETITIONS)} giải): {EXPANDED_COMPETITIONS}')
print(f'Phạm vi mùa giải ({len(EXPANDED_SEASONS)} mùa): {EXPANDED_SEASONS}')
print('-> Phạm vi Scope sẽ được kiểm tra và lọc theo yêu cầu nghiệp vụ.')

In [ ]:
# 4. Khử trùng lặp và đảm bảo tính vẹn toàn tham chiếu

# Khử trùng lặp theo Khóa chính (Primary Keys)
df_competitions_clean = df_competitions.drop_duplicates(subset=['competition_id']).copy()
df_clubs_clean        = df_clubs.drop_duplicates(subset=['club_id']).copy()
df_players_clean      = df_players.drop_duplicates(subset=['player_id']).copy()
df_games_clean        = df_games.drop_duplicates(subset=['game_id']).copy()
df_appearances_clean  = df_appearances.drop_duplicates(subset=['appearance_id']).copy()

# Đảm bảo tính vẹn toàn tham chiếu (Bổ sung CLB xuất hiện trong games nhưng thiếu ở clubs.csv)
all_game_club_ids = (
    set(df_games_clean['home_club_id'].dropna().astype(int)) |
    set(df_games_clean['away_club_id'].dropna().astype(int))
)
missing_cids = all_game_club_ids - set(df_clubs_clean['club_id'])

if missing_cids:
    print(f'Bổ sung {len(missing_cids)} CLB còn thiếu từ tệp trận đấu...')
    home_names = (df_games_clean.dropna(subset=['home_club_id', 'home_club_name'])
                  .drop_duplicates('home_club_id')
                  .set_index('home_club_id')['home_club_name'].to_dict())
    away_names = (df_games_clean.dropna(subset=['away_club_id', 'away_club_name'])
                  .drop_duplicates('away_club_id')
                  .set_index('away_club_id')['away_club_name'].to_dict())
    supp = []
    for cid in missing_cids:
        cname = home_names.get(cid, away_names.get(cid, f'Club_{cid}'))
        supp.append({
            'club_id': cid, 'club_code': f'club_{cid}',
            'name': str(cname).strip(), 'domestic_competition_id': 'OTH',
            'total_market_value': float('nan'), 'squad_size': 25,
            'average_age': float('nan'), 'foreigners_number': float('nan'),
            'foreigners_percentage': float('nan'), 'national_team_players': float('nan'),
            'stadium_name': 'Unknown Stadium', 'stadium_seats': 0,
            'net_transfer_record': '0', 'coach_name': 'Unknown Coach'
        })
    df_clubs_clean = pd.concat([df_clubs_clean, pd.DataFrame(supp)], ignore_index=True)
    df_clubs_clean = df_clubs_clean.drop_duplicates(subset=['club_id'])

print('Khử trùng lặp & Bảo đảm vẹn toàn tham chiếu hoàn tất:')
print(f'  - Competitions: {len(df_competitions_clean):,}')
print(f'  - Clubs       : {len(df_clubs_clean):,}')
print(f'  - Players     : {len(df_players_clean):,}')
print(f'  - Games       : {len(df_games_clean):,}')
print(f'  - Appearances : {len(df_appearances_clean):,}')

In [ ]:
# 5. Chọn lọc các cột cần thiết

# Chuẩn hóa kiểu dữ liệu, làm sạch chuỗi & xử lý NULL
df_appearances_clean['appearance_id']  = df_appearances_clean['appearance_id'].astype(str).str.strip()
df_appearances_clean['game_id']        = df_appearances_clean['game_id'].astype('int64')
df_appearances_clean['player_id']      = df_appearances_clean['player_id'].astype('int64')
df_appearances_clean['player_club_id'] = df_appearances_clean['player_club_id'].astype('int64')
df_appearances_clean['competition_id'] = df_appearances_clean['competition_id'].astype(str).str.strip()
df_appearances_clean['date']           = pd.to_datetime(df_appearances_clean['date'], errors='coerce').dt.strftime('%Y-%m-%d')
for col in ['goals', 'assists', 'yellow_cards', 'red_cards', 'minutes_played']:
    df_appearances_clean[col] = df_appearances_clean[col].fillna(0).astype('int64')

df_players_clean['player_id']              = df_players_clean['player_id'].astype('int64')
df_players_clean['name']                   = df_players_clean['name'].fillna('Unknown Player').astype(str).str.strip()
df_players_clean['date_of_birth']          = (pd.to_datetime(df_players_clean['date_of_birth'].fillna('1999-01-01'), errors='coerce')
                                              .fillna(pd.Timestamp('1999-01-01')).dt.strftime('%Y-%m-%d'))
df_players_clean['country_of_citizenship'] = df_players_clean['country_of_citizenship'].fillna('Unknown').astype(str).str.strip()
df_players_clean['position']               = df_players_clean['position'].fillna('Unknown').astype(str).str.strip()
df_players_clean['sub_position']           = df_players_clean['sub_position'].fillna('Unknown').astype(str).str.strip()
df_players_clean['foot']                   = df_players_clean['foot'].fillna('Unknown').astype(str).str.strip()
df_players_clean['height_in_cm']           = df_players_clean['height_in_cm'].fillna(180.0).astype('int64')

df_clubs_clean['club_id']       = df_clubs_clean['club_id'].astype('int64')
df_clubs_clean['name']          = df_clubs_clean['name'].fillna('Unknown Club').astype(str).str.strip()
df_clubs_clean['stadium_name']  = df_clubs_clean['stadium_name'].fillna('Unknown Stadium').astype(str).str.strip()
df_clubs_clean['stadium_seats'] = df_clubs_clean['stadium_seats'].fillna(0).astype('int64')
df_clubs_clean['coach_name']    = df_clubs_clean['coach_name'].fillna('Unknown Coach').astype(str).str.strip()
df_clubs_clean['squad_size']    = df_clubs_clean['squad_size'].fillna(25).astype('int64')

df_competitions_clean['competition_id'] = df_competitions_clean['competition_id'].astype(str).str.strip()
df_competitions_clean['name']           = df_competitions_clean['name'].fillna('Unknown Competition').astype(str).str.strip()
df_competitions_clean['country_name']   = df_competitions_clean['country_name'].fillna('Europe').astype(str).str.strip()
df_competitions_clean['type']           = df_competitions_clean['type'].fillna('domestic_league').astype(str).str.strip()

df_games_clean['game_id']        = df_games_clean['game_id'].astype('int64')
df_games_clean['competition_id'] = df_games_clean['competition_id'].astype(str).str.strip()
df_games_clean['season']         = df_games_clean['season'].fillna(2022).astype('int64')
df_games_clean['round']          = df_games_clean['round'].fillna('Regular Season').astype(str).str.strip()
df_games_clean['date']           = pd.to_datetime(df_games_clean['date'], errors='coerce').dt.strftime('%Y-%m-%d')
df_games_clean['home_club_id']   = df_games_clean['home_club_id'].fillna(0).astype('int64')
df_games_clean['away_club_id']   = df_games_clean['away_club_id'].fillna(0).astype('int64')
df_games_clean['home_club_goals'] = df_games_clean['home_club_goals'].fillna(0).astype('int64')
df_games_clean['away_club_goals'] = df_games_clean['away_club_goals'].fillna(0).astype('int64')
df_games_clean['stadium']        = df_games_clean['stadium'].fillna('Unknown Stadium').astype(str).str.strip()

# Trích xuất các cột thuộc tính cần thiết theo Mô hình Kho dữ liệu Star Schema
df_apps_clean = df_appearances_clean[[
    'appearance_id', 'game_id', 'player_id', 'player_club_id', 'date',
    'competition_id', 'goals', 'assists', 'minutes_played',
    'yellow_cards', 'red_cards'
]].copy()

df_players_out = df_players_clean[[
    'player_id', 'name', 'date_of_birth', 'country_of_citizenship',
    'position', 'sub_position', 'foot', 'height_in_cm'
]].copy()

df_clubs_out = df_clubs_clean[[
    'club_id', 'name', 'stadium_name', 'stadium_seats', 'coach_name', 'squad_size'
]].copy()

df_comps_out = df_competitions_clean[[
    'competition_id', 'name', 'country_name', 'type'
]].copy()

df_games_out = df_games_clean[[
    'game_id', 'competition_id', 'season', 'round', 'date', 'home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals', 'stadium'
]].copy()

print('=== KÍCH THƯỚC DỮ LIỆU SẠCH HOÀN CHỈNH ===')
print(f'  - Appearances : {df_apps_clean.shape[0]:,} dòng, {df_apps_clean.shape[1]} cột')
print(f'  - Players     : {df_players_out.shape[0]:,} dòng, {df_players_out.shape[1]} cột')
print(f'  - Clubs       : {df_clubs_out.shape[0]:,} dòng, {df_clubs_out.shape[1]} cột')
print(f'  - Competitions: {df_comps_out.shape[0]:,} dòng, {df_comps_out.shape[1]} cột')
print(f'  - Games       : {df_games_out.shape[0]:,} dòng, {df_games_out.shape[1]} cột')

In [ ]:
# 6. Xuất dữ liệu sạch - Tệp dữ liệu tiền xử lý
OUTPUT_DIR = 'cleaned_data' if os.path.exists('cleaned_data') else os.path.join('football', 'cleaned_data')
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_apps_clean.to_csv( os.path.join(OUTPUT_DIR, 'cleaned_appearances.csv'),  index=False, encoding='utf-8-sig')
df_players_out.to_csv(os.path.join(OUTPUT_DIR, 'cleaned_players.csv'),      index=False, encoding='utf-8-sig')
df_clubs_out.to_csv(  os.path.join(OUTPUT_DIR, 'cleaned_clubs.csv'),        index=False, encoding='utf-8-sig')
df_comps_out.to_csv(  os.path.join(OUTPUT_DIR, 'cleaned_competitions.csv'), index=False, encoding='utf-8-sig')
df_games_out.to_csv(  os.path.join(OUTPUT_DIR, 'cleaned_games.csv'),        index=False, encoding='utf-8-sig')

print('ĐÃ XUẤT THÀNH CÔNG TỆP DỮ LIỆU TIỀN XỬ LÝ VÀO THƯ MỤC:', os.path.abspath(OUTPUT_DIR))
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.csv'):
        fpath = os.path.join(OUTPUT_DIR, f)
        print(f'  - {f}: {os.path.getsize(fpath)/1024/1024:.2f} MB')